In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,RobustScaler
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
import datetime
import re

# --- GPU Überprüfung und Setup ---
#physical_devices = tf.config.list_physical_devices('GPU')
#if physical_devices:
#    print(f"NVIDIA GPU erkannt: {len(physical_devices)} GPU(s) verfügbar.")
#    try:
#        for gpu in physical_devices:
#            tf.config.experimental.set_memory_growth(gpu, True)
#        print("GPU Memory Growth aktiviert (verhindert Out-of-Memory Fehler).")
#    except RuntimeError as e:
#        print(e)
#else:
#    print("Keine GPU erkannt. Training wird auf der CPU ausgeführt.")

df = pd.read_csv('training_data_ready.csv')
#df = df.drop([92, 99, 31, 169, 150, 43], axis=0) #[21,43,92,159] # [92,99,31,169,150,43] #[17,21,43,53,62,92,129,150,159,166,169] #[15,17,21,30,35,43,53,62,67,72,73,75,91,92,129,139,142,150,159,166,169]
target_cols = ['x', 'y', 'z','Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]#,'A_1_center','A_1_front','A_1_left','A_1_right','A_1_back']] #
    
X = df[x_cols]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

KeyboardInterrupt: 

In [ ]:
model = tf.keras.models.load_model("Modelle2/model_NoDrop_noA1_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_68.3391.keras")
y_pred = model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred","Gewicht_pred"], index=y_test.index
)

results = y_test.copy()
results[["x_pred", "y_pred", "z_pred","Gewicht_pred"]] = y_pred_df

results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

#restliche_spalten = df.loc[y_test.index].drop(
#    columns=["x", "y", "z"], errors="ignore"
#)
#doppelte_spalten = [c for c in restliche_spalten.columns if c in results.columns]
#restliche_spalten = restliche_spalten.drop(columns=doppelte_spalten)

results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

model_name = model.__class__.__name__

try:
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)

    opt_name = model.optimizer.__class__.__name__
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"

    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    param_str = "keras_nn"

filename = f"Auswertung_test_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)

#results.to_csv(filename, index=True, sep=";", decimal=",")

print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")


2/2 [==============================] - 0s 2ms/step
Auswertung Testdatensatz:
  - MAE x: 14.3599
  - MAE y: 16.7147
  - MAE z: 3.8876
  - MAE Gewicht: 7.2646
  - MSE x: 487.3217
  - MSE y: 621.7423
  - MSE z: 34.3553
  - MSE Gewicht: 118.8430


In [ ]:
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
139,"81,39","46,35","20,00","58,10","52,55","61,79","15,32","53,61","28,84","15,44","4,68","4,49","831,502967","238,542077","21,892148","20,157149",Endboss_L_U_S_X_3.csv
30,"53,33","91,00","30,00","443,80","75,96","64,66","34,88","437,71","22,63","26,34","4,88","6,09","511,894505","693,685286","23,833365","37,129103",3eck_L_L_4.csv
119,"54,50","47,50","20,00","43,90","44,33","69,86","20,85","56,03","10,17","22,36","0,85","12,13","103,464247","499,917086","0,715856","147,147791",6eck_L_U_4.csv
29,"53,33","91,00","30,00","443,80","73,88","63,01","34,68","415,78","20,55","27,99","4,68","28,02","422,356009","783,586480","21,889586","785,300050",3eck_L_L_3.csv
144,"46,35","81,39","20,00","58,10","49,75","60,86","20,11","55,58","3,40","20,53","0,11","2,52","11,580709","421,357772","0,012453","6,350564",Endboss_L_L_S_Y_2.csv
163,"120,00","12,50","17,50","71,60","110,43","16,43","22,50","66,89","9,57","3,93","5,00","4,71","91,579929","15,468884","25,028351","22,169088",Rechteck_Lang_L_U_6.csv
166,"12,50","120,00","17,50","71,60","79,02","27,08","24,53","60,21","66,52","92,92","7,03","11,39","4425,000290","8635,046267","49,420588","129,754716",Rechteck_Lang_L_L_3.csv
51,"31,75","25,00","50,00","127,80","44,93","32,00","28,05","127,75","13,18","7,00","21,95","0,05","173,636095","49,014581","481,934673","0,002241",Rechteck_H_L_L_1.csv
105,"47,50","32,50","17,25","19,30","42,55","42,16","15,48","23,20","4,95","9,66","1,77","3,90","24,490537","93,342426","3,140502","15,180221",Durch_L_U_D_O_2.csv
60,"87,50","47,50","20,00","51,70","43,63","78,80","20,10","56,87","43,87","31,30","0,10","5,17","1924,634041","979,567929","0,009598","26,720804",Lumi_L_U_1.csv


In [ ]:
import os

model_paths = [
    #"Modelle2/model_Variant2_all_robust_layers_512-256-128-64_l2_0.0_drop_0.0_lr_0.005_valloss_156.5477.keras",
     "Modelle2/model_NoDrop_all_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_78.0956.keras",
     "Modelle2/model_NoDrop_noA1_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_82.0381.keras",
     "Modelle2/model_NoDrop_noA1_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_68.3391.keras",
    # "model_NoDrop_noA1_robust_layers_512-256-128-64_l2_0.0_drop_0.0_lr_0.005_valloss_96.1255.keras"
]

X_data = X_test.copy()
y_data = y_test.copy()

all_predictions = []

for path in model_paths:
    if not os.path.exists(path):
        print(f"Überspringe (nicht gefunden): {path}")
        continue

    print(f"Lade Modell: {path}")
    model = tf.keras.models.load_model(path)

    if 'noA1' in path:
        print(" -> 'noA1' erkannt. Entferne A_1_... Spalten für dieses Modell.")
        cols_to_drop = ['A_1_center', 'A_1_front', 'A_1_left', 'A_1_right', 'A_1_back']
        cols_to_drop = [c for c in cols_to_drop if c in X_data.columns]
        
        X_subset = X_data.drop(columns=cols_to_drop)
        scaler_subset = RobustScaler()
        X_scaled = scaler_subset.fit_transform(X_subset)
    else:
        scaler_full = RobustScaler()
        X_scaled = scaler_full.fit_transform(X_data)

    preds = model.predict(X_scaled)
    all_predictions.append(preds)

if len(all_predictions) > 0:
    all_predictions_array = np.array(all_predictions)

    ensemble_preds = np.mean(all_predictions_array, axis=0)

    results_ensemble = y_data.copy()
    pred_cols = ["x_pred", "y_pred", "z_pred", "Gewicht_pred"]
    results_ensemble[pred_cols] = ensemble_preds

    results_ensemble["abs_error_x"] = np.abs(results_ensemble["x_pred"] - results_ensemble["x"])
    results_ensemble["abs_error_y"] = np.abs(results_ensemble["y_pred"] - results_ensemble["y"])
    results_ensemble["abs_error_z"] = np.abs(results_ensemble["z_pred"] - results_ensemble["z"])
    results_ensemble["abs_error_Gewicht"] = np.abs(results_ensemble["Gewicht_pred"] - results_ensemble["Gewicht"])

    results_ensemble["mse_x"] = (results_ensemble["x_pred"] - results_ensemble["x"]) ** 2
    results_ensemble["mse_y"] = (results_ensemble["y_pred"] - results_ensemble["y"]) ** 2
    results_ensemble["mse_z"] = (results_ensemble["z_pred"] - results_ensemble["z"]) ** 2
    results_ensemble["mse_Gewicht"] = (results_ensemble["Gewicht_pred"] - results_ensemble["Gewicht"]) ** 2

    results_ensemble = pd.concat([results_ensemble, df.loc[y_data.index, ["source_file"]]], axis=1)

    print("\nAuswertung ENSEMBLE auf Trainingsdaten:")
    print(f"  - MAE x: {results_ensemble['abs_error_x'].mean():.4f}")
    print(f"  - MAE y: {results_ensemble['abs_error_y'].mean():.4f}")
    print(f"  - MAE z: {results_ensemble['abs_error_z'].mean():.4f}")
    print(f"  - MAE Gewicht: {results_ensemble['abs_error_Gewicht'].mean():.4f}")
    print(f"  - MSE x: {results_ensemble['mse_x'].mean():.4f}")
    print(f"  - MSE y: {results_ensemble['mse_y'].mean():.4f}")
    print(f"  - MSE z: {results_ensemble['mse_z'].mean():.4f}")
    print(f"  - MSE Gewicht: {results_ensemble['mse_Gewicht'].mean():.4f}")
else:
    print("Es wurden keine Modelle gefunden/geladen.")

Lade Modell: Modelle2/model_NoDrop_all_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_78.0956.keras
2/2 [==============================] - 0s 2ms/step
Lade Modell: Modelle2/model_NoDrop_noA1_minmax_layers_256-128-64_l2_0.2_drop_0.0_lr_0.01_valloss_82.0381.keras
 -> 'noA1' erkannt. Entferne A_1_... Spalten für dieses Modell.
2/2 [==============================] - 0s 3ms/step
Lade Modell: Modelle2/model_NoDrop_noA1_robust_layers_128-64-32-16_l2_0.0_drop_0.1_lr_0.01_valloss_68.3391.keras
 -> 'noA1' erkannt. Entferne A_1_... Spalten für dieses Modell.
2/2 [==============================] - 0s 4ms/step

Auswertung ENSEMBLE auf Trainingsdaten:
  - MAE x: 16.0780
  - MAE y: 17.6943
  - MAE z: 4.5065
  - MAE Gewicht: 31.5021
  - MSE x: 465.2242
  - MSE y: 625.6558
  - MSE z: 43.9067
  - MSE Gewicht: 2593.2844


In [ ]:
if len(all_predictions) > 0:
    display(results_ensemble.style.format(
        {
            "x": "{:.2f}",
            "x_pred": "{:.2f}",
            "abs_error_x": "{:.2f}",
            "y": "{:.2f}",
            "y_pred": "{:.2f}",
            "abs_error_y": "{:.2f}",
            "z": "{:.2f}",
            "z_pred": "{:.2f}",
            "abs_error_z": "{:.2f}",
            "Gewicht": "{:.2f}",
            "Gewicht_pred": "{:.2f}",
            "abs_error_Gewicht": "{:.2f}",
        },
        decimal=",",
    ).background_gradient(
        subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
    ))

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht,source_file
139,"81,39","46,35","20,00","58,10","54,78","55,64","19,10","56,71","26,61","9,29","0,90","1,39","707,989847","86,310042","0,805577","1,928848",Endboss_L_U_S_X_3.csv
30,"53,33","91,00","30,00","443,80","58,09","52,21","35,47","310,84","4,76","38,79","5,47","132,96","22,642932","1504,989431","29,891249","17679,076722",3eck_L_L_4.csv
119,"54,50","47,50","20,00","43,90","50,52","49,87","18,18","41,50","3,98","2,37","1,82","2,40","15,832988","5,609556","3,310490","5,771047",6eck_L_U_4.csv
29,"53,33","91,00","30,00","443,80","51,87","68,65","31,95","329,18","1,46","22,35","1,95","114,62","2,127161","499,453545","3,819855","13137,592171",3eck_L_L_3.csv
144,"46,35","81,39","20,00","58,10","49,43","41,73","29,19","50,70","3,08","39,66","9,19","7,40","9,478954","1572,784016","84,469993","54,700668",Endboss_L_L_S_Y_2.csv
163,"120,00","12,50","17,50","71,60","123,71","7,38","18,51","62,04","3,71","5,12","1,01","9,56","13,790713","26,176063","1,020127","91,385049",Rechteck_Lang_L_U_6.csv
166,"12,50","120,00","17,50","71,60","80,00","39,07","20,52","64,33","67,50","80,93","3,02","7,27","4556,845341","6549,434644","9,135743","52,890597",Rechteck_Lang_L_L_3.csv
51,"31,75","25,00","50,00","127,80","56,15","36,35","23,72","105,20","24,40","11,35","26,28","22,60","595,192731","128,804195","690,674527","510,956721",Rechteck_H_L_L_1.csv
105,"47,50","32,50","17,25","19,30","45,42","28,13","24,90","24,57","2,08","4,37","7,65","5,27","4,332329","19,120870","58,453936","27,741464",Durch_L_U_D_O_2.csv
60,"87,50","47,50","20,00","51,70","58,26","51,11","25,56","54,97","29,24","3,61","5,56","3,27","854,745707","13,049627","30,882212","10,686123",Lumi_L_U_1.csv
